# ETL 04 -  Inspect: esquemas, heads y reglas de limpieza

Objetivo: abrir los Parquet descargados, inspeccionar el `head` de cada tipo de record, comparar columnas y definir reglas de limpieza reproducibles.

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path.cwd()
BASE_DIR = PROJECT_DIR if PROJECT_DIR.name == "etl-04" else PROJECT_DIR / "etl-04"
BRONZE_DIR = BASE_DIR / "data" / "bronze" / "tlc"
SILVER_DIR = BASE_DIR / "data" / "silver" / "tlc"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

RECORD_TYPES = {"yellow", "green", "fhv", "fhvhv"}


def parse_tlc_filename(filename: str) -> dict | None:
    """Extrae tipo, año y mes desde nombres como yellow_tripdata_2026-01.parquet."""
    if not filename.endswith(".parquet") or "_tripdata_" not in filename:
        return None

    record_type, year_month_ext = filename.split("_tripdata_", 1)
    year_month = year_month_ext.removesuffix(".parquet")

    if record_type not in RECORD_TYPES or "-" not in year_month:
        return None

    year_text, month_text = year_month.split("-", 1)
    if not year_text.isdigit() or not month_text.isdigit():
        return None

    return {
        "record_type": record_type,
        "year": int(year_text),
        "month": int(month_text),
        "year_month": year_month,
    }


## 1. Inventario local

Leemos lo que efectivamente existe en la capa `bronze`. Esto permite trabajar aunque se haya descargado solo un subconjunto.

In [2]:
files = []
for path in sorted(BRONZE_DIR.glob("*_tripdata_2026-*.parquet")):
    parsed = parse_tlc_filename(path.name)
    if parsed is None:
        continue
    parquet_file = pq.ParquetFile(path)
    files.append(
        {
            **parsed,
            "filename": path.name,
            "path": path,
            "rows": parquet_file.metadata.num_rows,
            "columns": len(parquet_file.schema_arrow.names),
            "size_mb": round(path.stat().st_size / 1024**2, 2),
        }
    )

inventory = pd.DataFrame(files).sort_values(["year_month", "record_type"])
display(inventory.drop(columns="path"))


,record_type,year,month,year_month,filename,rows,columns,size_mb
0,fhv,2026,1,2026-01,fhv_tripdata_2026-01.parquet,1941722,7,20.57
1,fhvhv,2026,1,2026-01,fhvhv_tripdata_2026-01.parquet,20940373,25,482.43
2,green,2026,1,2026-01,green_tripdata_2026-01.parquet,40272,21,0.95
3,yellow,2026,1,2026-01,yellow_tripdata_2026-01.parquet,3724889,20,61.19


## 2. Head de cada tipo de record

Para no cargar todo el archivo, tomamos el primer row group y mostramos solo algunas filas.

In [4]:
def parquet_head(path: Path, n: int = 5) -> pd.DataFrame:
    parquet_file = pq.ParquetFile(path)
    table = parquet_file.read_row_group(0).slice(0, n)
    return table.to_pandas()


heads = {}
for record_type in sorted(inventory["record_type"].unique()):
    first_path = inventory.loc[inventory["record_type"].eq(record_type), "path"].iloc[0]
    print("=" * 80)
    print(record_type, first_path.name)
    heads[record_type] = parquet_head(first_path)
    display(heads[record_type])


fhv fhv_tripdata_2026-01.parquet


,dispatching_base_num,pickup_datetime,dropOff_datetime,PUlocationID,DOlocationID,SR_Flag,Affiliated_base_number
0,B00112,2026-01-01 00:02:38,2026-01-01 00:33:13,NaN,14,NaN,B00112
1,B00112,2026-01-01 00:46:51,2026-01-01 00:54:02,NaN,14,NaN,B00112
2,B00112,2026-01-01 00:46:13,2026-01-01 01:13:56,NaN,14,NaN,B00112
3,B00112,2026-01-01 00:17:44,2026-01-01 00:24:31,NaN,14,NaN,B00112
4,B00112,2026-01-01 00:35:25,2026-01-01 00:40:26,NaN,14,NaN,B00112


fhvhv fhvhv_tripdata_2026-01.parquet


,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,cbd_congestion_fee
0,HV0003,B03404,B03404,2026-01-01 00:50:37,2026-01-01 00:52:31,2026-01-01 00:54:30,2026-01-01 01:13:23,262,79,4.300,...,2.75,0.0,0.00,21.10,N,N,N,N,N,1.5
1,HV0003,B03404,B03404,2026-01-01 00:09:12,2026-01-01 00:12:17,2026-01-01 00:12:39,2026-01-01 00:22:42,195,52,1.890,...,0.00,0.0,0.00,9.10,N,N,N,N,N,0.0
2,HV0003,B03404,B03404,2026-01-01 00:16:16,2026-01-01 00:29:33,2026-01-01 00:31:34,2026-01-01 00:55:21,25,181,1.840,...,0.00,0.0,2.67,21.94,N,N,N,N,N,0.0
3,HV0005,B03406,NaN,2026-01-01 00:08:19,2026-01-01 00:09:08,2026-01-01 00:09:37,2026-01-01 00:22:12,141,226,3.307,...,2.75,0.0,0.00,13.42,N,N,N,N,Y,1.5
4,HV0005,B03406,NaN,2026-01-01 00:21:16,2026-01-01 00:26:30,2026-01-01 00:26:40,2026-01-01 00:39:06,226,226,2.060,...,0.00,0.0,2.98,11.39,N,N,N,N,Y,0.0


green green_tripdata_2026-01.parquet


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1,65,233,2,6.20,31.7,...,1.5,7.5,0.0,NaN,1.0,45.20,1,1,2.75,0.75
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5,66,188,5,5.36,50.0,...,0.0,10.2,0.0,NaN,1.0,61.20,1,2,0.00,0.00
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1,65,179,4,10.60,41.5,...,1.5,2.0,0.0,NaN,1.0,46.00,1,1,0.00,0.00
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1,42,141,1,4.20,19.8,...,1.5,0.0,0.0,NaN,1.0,25.05,2,1,2.75,0.00
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1,95,82,1,2.76,19.1,...,0.5,0.0,0.0,NaN,1.0,21.60,2,1,0.00,0.00


yellow yellow_tripdata_2026-01.parquet


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


## 3. Comparacion de esquemas

Miramos columnas por tipo y diferencias, para ver cómo normalizamos la data.

In [5]:
example_file = inventory["path"].iloc[0]
parquet_file = pq.ParquetFile(example_file)
print("File:", example_file)
print("Schema PyArrow:")
print(parquet_file.schema_arrow)
print("Filas:", parquet_file.metadata.num_rows)
print("Columnas:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.metadata.num_row_groups)

File: /Volumes/Data/flavioc/Codes/GitHub/cursos/Python/diplo-ia/etl-04/data/tlc/raw/fhv_tripdata_2026-01.parquet
Schema PyArrow:
dispatching_base_num: large_string
pickup_datetime: timestamp[us]
dropOff_datetime: timestamp[us]
PUlocationID: int64
DOlocationID: int64
SR_Flag: int64
Affiliated_base_number: large_string
Filas: 1941722
Columnas: 7
Row groups: 2


In [6]:
schemas = []
for row in inventory.itertuples(index=False):
    parquet_file = pq.ParquetFile(row.path)
    schema_arrow = parquet_file.schema_arrow
    for field in parquet_file.schema_arrow:
        schemas.append(
            {
                "record_type": row.record_type,
                "year_month": row.year_month,
                "column": field.name,
                "dtype": str(field.type),
            }
        )

schema_df = pd.DataFrame(schemas)
schema_df.head(10)

,record_type,year_month,column,dtype
0,fhv,2026-01,dispatching_base_num,large_string
1,fhv,2026-01,pickup_datetime,timestamp[us]
2,fhv,2026-01,dropOff_datetime,timestamp[us]
3,fhv,2026-01,PUlocationID,int64
4,fhv,2026-01,DOlocationID,int64
5,fhv,2026-01,SR_Flag,int64
6,fhv,2026-01,Affiliated_base_number,large_string
7,fhvhv,2026-01,hvfhs_license_num,large_string
8,fhvhv,2026-01,dispatching_base_num,large_string
9,fhvhv,2026-01,originating_base_num,large_string


In [8]:

schema_matrix = schema_df.pivot_table(
    index="column",
    columns="record_type",
    values="dtype",
    aggfunc=lambda values: sorted(set(values))[0],
    fill_value="",
)

display(schema_matrix)
schema_df.to_csv(SILVER_DIR / "tlc_2026_schema_long.csv", index=False)


record_type,fhv,fhvhv,green,yellow
column,,,,
Affiliated_base_number,large_string,,,
Airport_fee,,,,double
DOLocationID,,int32,int32,int32
DOlocationID,int64,,,
PULocationID,,int32,int32,int32
PUlocationID,int64,,,
RatecodeID,,,int64,int64
SR_Flag,int64,,,
VendorID,,,int32,int32


## 4. Perfil rapido de calidad

Cargamos una muestra por tipo para detectar problemas frecuentes: fechas fuera de rango, montos negativos, viajes sin distancia y ubicaciones nulas.

In [9]:
SAMPLE_ROWS = 100_000

DATETIME_COLUMNS = {
    "yellow": ("tpep_pickup_datetime", "tpep_dropoff_datetime"),
    "green": ("lpep_pickup_datetime", "lpep_dropoff_datetime"),
    "fhv": ("pickup_datetime", "dropOff_datetime"),
    "fhvhv": ("pickup_datetime", "dropoff_datetime"),
}

def read_sample(path: Path, n: int = SAMPLE_ROWS) -> pd.DataFrame:
    parquet_file = pq.ParquetFile(path)
    table = parquet_file.read_row_group(0).slice(0, n)
    return table.to_pandas()


profiles = []
for row in inventory.itertuples(index=False):
    df = read_sample(row.path)
    pickup_col, dropoff_col = DATETIME_COLUMNS[row.record_type]
    pickup = pd.to_datetime(df[pickup_col], errors="coerce") if pickup_col in df else pd.Series(dtype="datetime64[ns]")
    dropoff = pd.to_datetime(df[dropoff_col], errors="coerce") if dropoff_col in df else pd.Series(dtype="datetime64[ns]")
    duration_min = (dropoff - pickup).dt.total_seconds() / 60

    profiles.append(
        {
            "filename": row.filename,
            "record_type": row.record_type,
            "sample_rows": len(df),
            "pickup_min": pickup.min(),
            "pickup_max": pickup.max(),
            "duration_min_p01": duration_min.quantile(0.01) if len(duration_min) else None,
            "duration_min_p50": duration_min.quantile(0.50) if len(duration_min) else None,
            "duration_min_p99": duration_min.quantile(0.99) if len(duration_min) else None,
            "trip_distance_zero_or_negative": int((df.get("trip_distance", pd.Series(dtype=float)) <= 0).sum()),
            "total_amount_negative": int((df.get("total_amount", pd.Series(dtype=float)) < 0).sum()),
            "pickup_location_nulls": int(df.get("PULocationID", pd.Series(dtype="object")).isna().sum()),
            "dropoff_location_nulls": int(df.get("DOLocationID", pd.Series(dtype="object")).isna().sum()),
        }
    )

profile_df = pd.DataFrame(profiles)
profile_df.to_csv(SILVER_DIR / "tlc_2026_quality_profile_sample.csv", index=False)
display(profile_df)

,filename,record_type,sample_rows,pickup_min,pickup_max,duration_min_p01,duration_min_p50,duration_min_p99,trip_distance_zero_or_negative,total_amount_negative,pickup_location_nulls,dropoff_location_nulls
0,fhv_tripdata_2026-01.parquet,fhv,100000,2026-01-01 00:00:00,2026-01-02 22:59:55,1.900000,14.500000,76.000000,0,0,0,0
1,fhvhv_tripdata_2026-01.parquet,fhvhv,100000,2026-01-01 00:00:00,2026-01-01 01:59:59,3.066667,15.983333,57.816833,0,0,0,0
2,green_tripdata_2026-01.parquet,green,40272,2025-12-27 16:49:41,2026-02-01 21:08:36,0.066667,12.883333,82.886500,1290,120,0,0
3,yellow_tripdata_2026-01.parquet,yellow,100000,2025-12-31 23:57:29,2026-01-02 14:59:59,0.000000,11.216667,58.900000,1805,1981,0,0


## 5. Reglas de limpieza propuestas

- Normalizar nombres de columnas a `snake_case`.
- Crear columnas comunes: `record_type`, `year_month`, `pickup_datetime`, `dropoff_datetime`, `pickup_location_id`, `dropoff_location_id`.
- Conservar columnas de negocio comparables cuando existan: distancia, pasajeros, montos, forma de pago, base/licencia.
- Filtrar viajes con fechas nulas, duracion negativa o duracion mayor a 24 horas.
- Para taxis amarillos y verdes: filtrar distancia negativa y montos negativos; conservar distancia cero para discutir casos especiales.
- No forzar que FHV tenga las mismas columnas de tarifa que taxi: es otro producto de datos.

Estas reglas se implementan en el siguiente notebook.